<a href="https://colab.research.google.com/github/K415mm/mcp_course_workshops/blob/main/Workshop_03_Network_Analysis/03_Network_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📡 Workshop 3: Network Analysis with MCP

Welcome to the third workshop of the RAISEGUARD Advanced Agentic AI Course!

**Goal:** Build an MCP-powered network analysis workflow that parses a network log file, identifies suspicious hosts and protocols, enriches external IPs, and produces a network risk report.

### Setup & Libraries
Run the cell below to install our dependencies.

In [ ]:
!pip install groq langchain-groq mcp requests -q
print("✅ Libraries installed successfully!")

### API Keys
For this workshop, you will need:
1. **Groq API Key** (For our LLM Brain)
2. **AbuseIPDB API Key** (For external IP enrichment)

Add them to the Google Colab **Secrets (🔑)** tab on the left menu, then run this cell.

In [ ]:
import os
from google.colab import userdata

keys = ["GROQ_API_KEY", "ABUSEIPDB_KEY"]
for k in keys:
    try:
        os.environ[k] = userdata.get(k)
        print(f"✅ {k} successfully loaded!")
    except userdata.SecretNotFoundError:
        print(f"⚠️ WARNING: {k} missing from Colab Secrets.")
        os.environ[k] = "" 

--- 
## 💾 Lab Data Setup
First, we need to generate our synthetic Network Event flow logs to simulate a digested PCAP file. Run this cell to create our `sample_network_events.json` file.

In [ ]:
import json

network_logs = [
  {"src_ip": "192.168.1.55", "dst_ip": "185.220.101.45", "dst_port": 443, "protocol": "TCP", "bytes": 48200, "packets": 340, "timestamp": "2026-03-09T21:10:00Z"},
  {"src_ip": "192.168.1.55", "dst_ip": "185.220.101.45", "dst_port": 9001, "protocol": "TCP", "bytes": 12400, "packets": 98, "timestamp": "2026-03-09T21:12:00Z"},
  {"src_ip": "192.168.1.102", "dst_ip": "8.8.8.8", "dst_port": 53, "protocol": "UDP", "bytes": 450, "packets": 12, "timestamp": "2026-03-09T21:14:00Z"},
  {"src_ip": "192.168.1.77", "dst_ip": "204.79.197.200", "dst_port": 80, "protocol": "TCP", "bytes": 3200, "packets": 22, "timestamp": "2026-03-09T21:16:00Z"},
  {"src_ip": "192.168.1.55", "dst_ip": "91.108.4.0", "dst_port": 4444, "protocol": "TCP", "bytes": 92100, "packets": 712, "timestamp": "2026-03-09T21:22:00Z"}
]

with open("sample_network_events.json", "w") as f:
    json.dump(network_logs, f, indent=2)

print("✅ Created sample_network_events.json with 5 flow events!")

--- 
## 🟢 Beginner Profile: Building the Flow Parsing Scripts
Before we make an autonomous agent, we need to build the raw Python functions that isolate external outbound traffic and high-risk network ports.

In [ ]:
LOG_PATH = "sample_network_events.json"
SUSPICIOUS_PORTS = {4444, 4445, 9001, 9002, 1337, 31337, 8080, 8443, 2222}

# Private RFC1918 ranges (simplified check)
def is_private_ip(ip: str) -> bool:
    return ip.startswith("192.168.") or ip.startswith("10.") or ip.startswith("172.")

def load_events() -> list:
    with open(LOG_PATH) as f:
        return json.load(f)

def get_suspicious_port_connections() -> dict:
    """Find all connections using non-standard or commonly abused ports."""
    try:
        events = load_events()
        hits = [e for e in events if e.get("dst_port") in SUSPICIOUS_PORTS]
        return {"suspicious_ports_checked": sorted(SUSPICIOUS_PORTS), "hit_count": len(hits), "events": hits, "status": "ok"}
    except Exception as e:
        return {"status": "error", "reason": str(e)}

# Let's test it!
print("Testing Suspicious Ports:\n", json.dumps(get_suspicious_port_connections(), indent=2))

--- 
## 🟡 Intermediate Profile: Exposing Tools via FastMCP
Now we will officially expose our network parsing tools alongside our external IP intelligence enrichment.

In [ ]:
from mcp.server.fastmcp import FastMCP
import requests

mcp = FastMCP("Network Analysis Server")
ABUSE_KEY = os.environ.get("ABUSEIPDB_KEY", "")

@mcp.tool()
def network_get_unique_external_ips() -> dict:
    """Extract all unique external (non-private) destination IP addresses from network logs."""
    try:
        events = load_events()
        external = sorted(set(e["dst_ip"] for e in events if not is_private_ip(e["dst_ip"]))
        )
        return {"external_ips": external, "count": len(external), "status": "ok"}
    except Exception as e:
        return {"status": "error", "reason": str(e)}

@mcp.tool()
def network_get_high_volume_flows(min_bytes: int = 50000) -> dict:
    """Find network flows that transferred more than a threshold of bytes."""
    try:
        events = load_events()
        hits = [e for e in events if e.get("bytes", 0) >= min_bytes]
        hits.sort(key=lambda x: x["bytes"], reverse=True)
        return {"threshold_bytes": min_bytes, "flow_count": len(hits), "flows": hits, "status": "ok"}
    except Exception as e:
        return {"status": "error", "reason": str(e)}

@mcp.tool()
def enrich_ip(ip_address: str) -> dict:
    """Check an IPv4 address for threat intelligence using AbuseIPDB."""
    try:
        resp = requests.get(
            "https://api.abuseipdb.com/api/v2/check",
            headers={"Key": ABUSE_KEY, "Accept": "application/json"},
            params={"ipAddress": ip_address, "maxAgeInDays": 90},
            timeout=10
        ).json().get("data", {})
        return {"ip": ip_address, "abuse_confidence": resp.get("abuseConfidenceScore", 0)}
    except Exception as e:
        return {"status": "error", "reason": str(e)}

print("✅ FastMCP Server initialized with tools:", [t.name for t in mcp._tool_manager.get_tools()])

--- 
## 🔴 Advanced Profile: The Autonomous Network Agent
We now provide the agent with our network analysis toolkit and command it to write a full risk report.

In [ ]:
from langchain.agents import initialize_agent, AgentType
from langchain.tools import tool
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.0)

# Langchain compatible tools
@tool
def ai_get_external_ips(query: str) -> str:
    """Extract all unique external (non-private) IPs from network logs. Query input can be 'all'."""
    return str(network_get_unique_external_ips())

@tool
def ai_get_suspicious_ports(query: str) -> str:
    """Find all connections using non-standard or commonly abused ports. Query input can be 'all'."""
    return str(get_suspicious_port_connections())

@tool
def ai_get_high_volume(query: str) -> str:
    """Find network flows that transferred more than 50K bytes. Query input can be 'all'."""
    return str(network_get_high_volume_flows(50000))

@tool
def ai_enrich_ip(ip: str) -> str:
    """Pass an IP to AbuseIPDB to get its threat score."""
    return str(enrich_ip(ip))

tools = [ai_get_external_ips, ai_get_suspicious_ports, ai_get_high_volume, ai_enrich_ip]

agent = initialize_agent(
    tools, llm, agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

print("🚀 Agent Pipeline Ready!")

In [ ]:
prompt = f"""
Analyze the network connections in our log file.
1. Identify all suspicious port connections.
2. Find high-volume data transfers that could be exfiltration.
3. Get all unique external IPs.
4. Enrich EACH external IP with AbuseIPDB threat intel using your tool.
5. Produce a NETWORK RISK REPORT with HIGH-RISK FINDINGS and RECOMMENDED ACTIONS.
"""

# Kick off the autonomous loop!
agent.run(prompt)